# Model Optimization — Quantization, LoRA, Memory/GPU/Speed

Optimization makes models fit and fly: fewer bits, adapters instead of full finetunes, KV-cache tactics, and decoding tricks.


## Learning Objectives

- Compare INT8/INT4/GPTQ/AWQ/GGUF quants
- Estimate LoRA parameter counts
- Apply memory and speed levers
- Measure prefix-cache savings


## 1. Quantization Overview

| Method | Idea | When |
|--------|------|------|
| INT8 | 8-bit weights/acts | Mild savings |
| INT4 / NF4 | 4-bit (QLoRA uses NF4) | Finetune + serve small |
| GPTQ | Post-train weight quant | GPU serving |
| AWQ | Activation-aware | Often strong 4-bit quality |
| GGUF quants | llama.cpp family | Local/CPU/GPU |

### Pitfalls
Quality cliffs on math, code, or multilingual tasks—eval per domain.


In [ ]:
# Demo 1 — Toy quality/size trade-off simulation
import numpy as np

rng = np.random.default_rng(0)
true = rng.normal(size=1000)
rows = []
for bits, noise in [(16, 0.01), (8, 0.03), (4, 0.08), (2, 0.2)]:
    q = true + rng.normal(scale=noise, size=true.shape)
    mse = float(np.mean((q - true) ** 2))
    size = 7 * bits / 16  # relative to FP16 7B
    rows.append((bits, round(mse, 4), round(size, 2)))
print("bits | mse | rel_size")
for r in rows:
    print(r)


## 2. LoRA & QLoRA

### Definition
**LoRA** trains low-rank adapters on frozen weights. **QLoRA** loads base in 4-bit and trains adapters—popular for consumer GPUs.

### When to use
Domain adaptation without full finetune cost.


In [ ]:
# Demo 2 — Parameter count for LoRA adapters
def lora_params(d_model: int, r: int, layers: int, matrices_per_layer: int = 2) -> int:
    # each matrix: d*r + r*d = 2*d*r
    return layers * matrices_per_layer * (2 * d_model * r)

print("LoRA params ~", lora_params(4096, 16, 32))
print("vs full 7B=", 7_000_000_000)


## 3. Memory & GPU Optimization

Levers: quantize weights, limit context, cap concurrency, CPU offload, tensor parallel, prefix/KV cache, chunked prefill.


## 4. Speed Optimization

- Continuous batching
- Speculative decoding
- FlashAttention kernels
- Smaller draft models
- Prompt caching / prefix reuse


In [ ]:
# Demo 3 — Prefix cache savings estimator
def cached_prefill_saved(system_tokens: int, requests: int, cost_per_1k: float = 0.0) -> dict:
    uncached = system_tokens * requests
    cached = system_tokens + requests  # pay system once + tiny per req overhead toy
    saved = uncached - cached
    return {"uncached_tokens": uncached, "cached_tokens": cached, "saved": saved}

print(cached_prefill_saved(2000, 100))


In [ ]:
# Demo 4 — Choose optimization stack
def optimize(goal: str, vram_gb: int) -> list[str]:
    acts = []
    if vram_gb < 16:
        acts.append("use_q4_or_awq")
    if goal == "throughput":
        acts += ["vllm_continuous_batching", "raise_max_num_seqs"]
    if goal == "latency":
        acts += ["limit_concurrency", "speculative_decoding"]
    acts.append("eval_task_suite")
    return acts

print(optimize("latency", 12))
print(optimize("throughput", 48))


### Try it yourself — Optimization

- A/B Q4 vs Q5 on a 50-prompt coding set
- Train a tiny LoRA on a style dataset (or simulate param math)
- Estimate prefix cache savings for your system prompt size


## Glossary / Key Terms

| Term | Meaning |
|------|--------|
| `LoRA` | Low-Rank Adaptation |
| `QLoRA` | Quantized base + LoRA training |
| `Speculative decoding` | Draft tokens verified by larger model |


## Deep Dive Workshop — 04 Model Optimization

This section expands the notebook into instructor/textbook depth. Work through each subsection: **definition → why it matters → how it works → intuition → pitfalls → when to use**.

```mermaid
flowchart TB
  D[Definition] --> W[Why it matters]
  W --> H[How it works]
  H --> I[Intuition]
  I --> P[Pitfalls]
  P --> U[When to use]
```


### Concept card pack for `04-model-optimization`

| Concept | Definition | Why it matters | Common pitfall |
|---------|------------|----------------|----------------|
| Primary abstraction | Core object this lesson centers on | Anchors design conversations | Vague naming |
| Quality oracle | How you know the system is right | Prevents demo-driven development | Using vibes only |
| Latency budget | Max user-visible wait | Drives architecture | Ignoring TTFT vs e2e |
| Cost unit | $ per successful task | Makes tradeoffs real | Optimizing tokens not outcomes |
| Trust boundary | Where data/control changes hands | Security design | Treating vendors as internal |
| Feedback loop | How production improves the system | Sustainable quality | No path from thumbs-down to evals |

**Intuition:** If you cannot fill this table for your system, you are not ready to choose models or frameworks.


### Pipeline walkthrough (apply to 04-model-optimization)

```
1. Input arrives (user / job / webhook)
2. Normalize + authorize + budget check
3. Gather context (files, RAG, tools, memory)
4. Model / deterministic compute
5. Validate output (schema, policy, tests)
6. Side effects (write, ticket, PR) with authz
7. Observe (metrics, traces, feedback)
8. Learn (eval suite growth, prompt/model revision)
```

**When to compress steps:** tiny internal tools. **When to keep all steps:** multi-tenant or regulated production.


### Local serving advanced notes

**Prefill vs decode** dominate different bottlenecks—size batches for your goal.  
**Quantization** must be validated on *your* tasks (code/math often sensitive).  
**OpenAI-compatible APIs** let you swap Ollama ↔ vLLM ↔ cloud with a router.

| Runtime | Best default use |
|---------|------------------|
| Ollama | Dev laptop UX |
| llama.cpp | Portable/edge |
| vLLM | Multi-user GPU |
| TGI | HF-centric prod features |


In [ ]:
# Extra demo — VRAM + context interaction sketch
def kv_cache_gb(layers, heads, dim, seq_len, bytes_per=2, batch=1):
    # very rough educational estimate
    return batch * layers * seq_len * heads * dim * bytes_per * 2 / (1024**3)

print('KV ~', round(kv_cache_gb(32, 32, 128, 8192), 2), 'GB')


In [ ]:
# Extra demo — OpenAI-compatible client switch
import os

def client_conf(profile: str) -> dict:
    profiles = {
        'ollama': {'base_url': 'http://localhost:11434/v1', 'api_key': 'ollama'},
        'vllm': {'base_url': os.getenv('VLLM_BASE_URL','http://localhost:8000/v1'), 'api_key': os.getenv('VLLM_API_KEY','YOUR_VLLM_API_KEY_HERE')},
        'cloud': {'base_url': 'https://api.openai.com/v1', 'api_key': os.getenv('OPENAI_API_KEY','sk-YOUR_OPENAI_API_KEY_HERE')},
    }
    return profiles[profile]

print(client_conf('ollama'))


### Sample interview Q&A — local serving

**Q:** When is self-hosting cheaper than APIs?  
**A:** When sustained high token volume amortizes GPUs + eng ops; include power, idle capacity, and oncall—not only GPU sticker price.

**Q:** Users complain about slow first token on a 70B local model.  
**A:** Check prefill length, quantization, batching contention, GPU util, and whether a smaller draft/cascade can handle easy prompts.


### Comparison matrix exercise

Fill this for two competing designs in this topic:

| Dimension | Option A | Option B | Winner / why |
|-----------|----------|----------|--------------|
| Latency | | | |
| Cost at 10× scale | | | |
| Quality risk | | | |
| Ops burden | | | |
| Security / privacy | | | |
| Time to MVP | | | |


In [ ]:
# Workshop demo — decision scorecard
from dataclasses import dataclass

@dataclass
class Option:
    name: str
    latency: int  # 1=best .. 5=worst
    cost: int
    quality_risk: int
    ops: int
    security: int

def score(o: Option, weights=None) -> float:
    weights = weights or dict(latency=1, cost=1, quality_risk=2, ops=1, security=2)
    return (
        o.latency*weights['latency'] + o.cost*weights['cost'] +
        o.quality_risk*weights['quality_risk'] + o.ops*weights['ops'] +
        o.security*weights['security']
    )

a = Option('A', 2, 3, 2, 2, 2)
b = Option('B', 3, 1, 3, 4, 2)
print(a.name, score(a), b.name, score(b), '-> prefer', a.name if score(a)<score(b) else b.name)


In [ ]:
# Workshop demo — experiment log (use while studying this notebook)
from dataclasses import dataclass, asdict
import json, time

@dataclass
class Experiment:
    hypothesis: str
    setup: str
    metric: str
    baseline: float | None = None
    treatment: float | None = None
    notes: str = ''
    ts: float = 0.0

    def __post_init__(self):
        if not self.ts:
            self.ts = time.time()

exp = Experiment(
    hypothesis='Technique from this lesson improves the primary metric',
    setup='Describe fixtures / model / dataset version',
    metric='name of metric',
    baseline=0.0,
    treatment=0.0,
)
print(json.dumps(asdict(exp), indent=2))


### ASCII architecture sketch template

```
[ Clients ]
     |
[ Edge / API Gateway ] -- authn/z, rate limit
     |
[ Orchestration ] ------+-- prompts / policies
     |                  +-- eval hooks
     +-- context layer (RAG / tools / memory)
     |
[ Model interface ] ---- local and/or cloud
     |
[ Data plane ] --------- indexes, OLTP, object store
     |
[ Observability ] ------ logs, metrics, traces, feedback
```

Copy into your notes and annotate trust boundaries with `***`.


### Pitfalls clinic (read aloud)

1. **Metric theater** — optimizing a proxy that users don't feel  
2. **Context stuffing** — more tokens ≠ more truth  
3. **Prompt as security** — never the only control  
4. **Hidden coupling** — tools/models/indexes version-drift  
5. **No rollback** — can't revert prompt/model quickly  
6. **Eval contamination** — testing on training-like snippets  
7. **Happy-path demos** — skipping adversarial & empty-retrieve cases  


### Try it yourself — extended set

1. Teach the top 3 ideas from this notebook to a rubber duck in 5 minutes  
2. Write 5 quiz questions (with answers) for a junior engineer  
3. Implement one code demo with a real dependency (API or local model) using env placeholders  
4. Break a naive design on purpose; list the failure mode and the fix  
5. Add two rows to your personal glossary with examples from work  
6. Produce a one-page cheat sheet you could use in an interview  


### Mini case study

**Scenario:** Leadership wants this capability in production in six weeks with two engineers.

**Your job:** Propose an MVP that keeps irreversible risks controlled, names the eval gates, and lists what you explicitly defer.

Deliverable structure:
- MVP user story  
- Non-goals  
- Architecture (6 boxes max)  
- Eval gate table  
- Risk register (top 5)  
- Week-by-week plan  


In [ ]:
# Case study helper — risk register
import pandas as pd

risks = pd.DataFrame([
    {'risk': 'quality_miss', 'likelihood': 3, 'impact': 3, 'mitigation': 'golden evals + canary'},
    {'risk': 'cost_overrun', 'likelihood': 3, 'impact': 2, 'mitigation': 'budgets + cache'},
    {'risk': 'data_leak', 'likelihood': 2, 'impact': 5, 'mitigation': 'ACL + redaction'},
    {'risk': 'prompt_injection', 'likelihood': 4, 'impact': 4, 'mitigation': 'boundaries + allowlists'},
    {'risk': 'ops_pages', 'likelihood': 3, 'impact': 3, 'mitigation': 'runbooks + rollback'},
])
risks['score'] = risks.likelihood * risks.impact
print(risks.sort_values('score', ascending=False).to_string(index=False))


### Interview drill (topic-local)

Use the STAR or design template. Timebox 8 minutes.

**Prompt:** “Walk me through how you would productionize the main idea of this notebook.”

Checklist for a strong answer:
- [ ] Clarifying questions  
- [ ] Constraints & numbers  
- [ ] Diagram  
- [ ] Deep dive on hardest part  
- [ ] Evals  
- [ ] Security  
- [ ] Rollout / rollback  


### Glossary boost

| Term | Expanded meaning |
|------|------------------|
| Canary | Partial traffic to a new variant with automatic rollback |
| Golden set | Versioned labeled examples for regression |
| TTFT | Time to first token — interactive UX driver |
| Packing | Selecting/ordering context under a token budget |
| HITL | Human approval inserted before side effects |
| Idempotency | Safe retries without duplicate side effects |
| Shadow traffic | New system sees traffic but doesn't affect users |
| Circuit breaker | Stop calling a failing dependency temporarily |


In [ ]:
# Self-check quiz (run and answer mentally before printing answers)
QUESTIONS = [
    'What oracle proves success for this topic?',
    'Name one metric that can be gamed and a better alternative.',
    'What is the top security failure mode?',
    'What would you defer in an MVP?',
    'How do you rollback a bad change here?',
]
for i, q in enumerate(QUESTIONS, 1):
    print(f'Q{i}. {q}')
print('\n--- suggested answer hints ---')
HINTS = [
    'executable tests / task success / human rubric',
    'longer answers != better; use task success',
    'trust boundary crossing / injection / ACL',
    'multi-agent, perfect UI, every connector',
    'versioned prompts/models + traffic switch',
]
for h in HINTS:
    print('-', h)


### Further practice roadmap for `04-model-optimization`

| Horizon | Action |
|---------|--------|
| Today | Re-run all code cells; note questions |
| This week | Apply one technique to a real repo/service |
| This month | Add an eval or security test covering this topic |
| Interview ready | Give a 10-minute teach-back with a diagram |


## Lab: End-to-end scenario

Work this scenario in your notes, then implement the smallest possible spike.

### Scenario brief
A team wants to adopt the techniques from this notebook for a **real internal tool** used daily by 200 people. Leadership cares about reliability and auditability more than flashy demos.

### Deliverables
1. One-paragraph problem statement  
2. Success metrics (3) with oracles  
3. Architecture sketch with trust boundaries  
4. Threats / failure modes (5)  
5. Eval plan (offline + online)  
6. 2-week MVP scope and explicit non-goals  

### Review questions
- What happens when context is empty?  
- What happens when the model is down?  
- What happens when a user is malicious?  
- How do you prove a release is safer/better than last week?  


In [ ]:
# Lab helper — MVP scope tracker
from dataclasses import dataclass, field

@dataclass
class MVP:
    must: list[str] = field(default_factory=list)
    should: list[str] = field(default_factory=list)
    defer: list[str] = field(default_factory=list)

    def show(self):
        for label, items in [('MUST', self.must), ('SHOULD', self.should), ('DEFER', self.defer)]:
            print(label)
            for i in items:
                print(' -', i)

mvp = MVP(
    must=['core happy path', 'authn', 'basic eval smoke', 'rollback switch'],
    should=['streaming UX', 'dashboards'],
    defer=['multi-agent', 'perfect personalization', 'every connector'],
)
mvp.show()


## Operator runbook sketch

| Symptom | Likely cause | First checks | Mitigation |
|---------|--------------|--------------|------------|
| Latency spike | Downstream model / retrieve | p95 by stage, saturation | shed load, failover |
| Quality drop | Prompt/model/index change | diff versions, eval slice | rollback |
| Cost spike | loops / huge prompts | tokens/req, step counts | budget breaker |
| Security alert | injection / ACL | traces + retrieved IDs | kill switch |

Keep this table in your ops wiki; customize per system.


In [ ]:
# Operator helper — stage latency rollup
from statistics import mean

stages = {
    'gateway': [20, 25, 22],
    'retrieve': [80, 120, 95],
    'generate': [900, 1100, 980],
}
for k, v in stages.items():
    print(f'{k:10} mean={mean(v):.0f}ms max={max(v)}ms')
print('e2e~', sum(mean(v) for v in stages.values()), 'ms')


## Teaching notes (for study groups)

- Start with the comparison table; argue both sides for 5 minutes  
- Pair-program one demo cell with a real endpoint (placeholder keys)  
- Each person writes one failure case the suite must catch  
- End with a 60-second summary of when *not* to use the technique  


## Summary & Key Takeaways

- Quantization and adapters are the main fit/adapt levers
- Memory and speed knobs differ for latency vs throughput
- Always close the loop with evaluation
